In [80]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import pickle
from sklearn.metrics import mean_squared_error

In [89]:
df = pd.read_csv("cleaned_data.csv", low_memory=False)

# Baseline model
Hier gaan we een baseline model maken met het gemiddelde en de mediaan

In [90]:
median = df[["stm_progfh_t_fh"]].median()

baseline = [median] * len(df)

print(f"R2 score: {r2_score(df['stm_progfh_t_fh'], baseline)}")

RMSE = np.sqrt(mean_squared_error(df['stm_progfh_t_fh'], baseline))
print(f"Root Mean Squared Error: {RMSE}")

baseline_info = {'baseline': baseline, 'rmse': RMSE}

with open('base_model.pkl', 'wb') as file:
    pickle.dump(baseline_info, file)

R2 score: -0.11372656130786285
Root Mean Squared Error: 70.92134786646736


## Conclusie
Met een score Baseline van -0.11372656130786285, kunnen we vastellen dat het gemiddelde een betere manier is om de stm_aann_t_fh te voorspellen dan de mediaan. Dit kunnen we zeggen omdat de score Baseline een negatief getal is. Een RMSE van 70.92134786646736 betekent dat de voorspelling er gemiddeld ~71 minuten naast de werkelijkheid zit.

# Model
Voor dit model gaan we gebruik maken van Decision trees met bins. Eerst moeten we onze nominale waardes (zie [cleaned_features.ipynb](cleaned_features.ipynb)) omzetten met get_dummies naar bins als dat nodig is.

In [91]:
def dummies_converter(df, value):
    dummies = pd.get_dummies(df[value])
    
    top_20_dummies = dummies.sum().nlargest(20).index
    
    df[value] = dummies[top_20_dummies].idxmax(axis=1)
    df.loc[~dummies[top_20_dummies].any(axis=1), value] = 9999
    
    dummies = pd.get_dummies(df[value], prefix='Index')
    dummies = dummies.astype(int)
    
    df = pd.concat([df, dummies], axis=1)
    
    return df

# Pas de functie toe op de opgegeven kolommen
for i in ['stm_geo_mld', 'stm_equipm_soort_mld', 'stm_equipm_nr_mld', 'stm_prioriteit', 'stm_oorz_groep', 'stm_oorz_code', 'stm_contractgeb_gst', 'stm_techn_gst']:
    df = dummies_converter(df, i)

Nu kunnen we het model maken met het gekozen algoritme

In [92]:
model = LinearRegression()

target = df["stm_progfh_t_fh"]
features = df.drop(columns=["stm_progfh_t_fh", 'stm_geo_mld', 'stm_equipm_soort_mld', 'stm_equipm_nr_mld', 'stm_prioriteit', 'stm_oorz_groep', 'stm_oorz_code', 'stm_contractgeb_gst', 'stm_techn_gst'])

X_train, X_test, y_train, y_test = train_test_split(features, target, test_size=0.2, random_state=10)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

RMSE = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Root Mean Square Error: {RMSE}")

model_info = {'model': model, 'rmse': RMSE}

with open('model.pkl', 'wb') as file:
    pickle.dump(model_info, file)

Root Mean Square Error: 59.871659967304744


## Conclusie
Met een RMSE van 59.871659967304744 kunnen we zeggen dat het model nog